In [2]:
from typing import TypedDict,List,Union
from langchain_core.messages import HumanMessage,AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv

In [3]:
!uv pip install langchain-ollama

Checked 1 package in 371ms


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:0.6b",
    temperature=0,
    
)

In [5]:
class AgentState(TypedDict):
    messages:List[Union[HumanMessage,AIMessage]]


In [6]:
def process(state:AgentState)->AgentState:
    response = llm.invoke(state["messages"])
    print(f"AI says : {response.content}")
    state["messages"].append(AIMessage(content=response.content))
    print("Current State : ", state["messages"])
    return state


In [7]:
graph=StateGraph(AgentState)
graph.add_node("process",process)

graph.add_edge(START,"process")
graph.add_edge("process",END)


agent=graph.compile()

In [10]:
conversation_history=[]
user_input=input("Enter: ")
while user_input!="exit":
    conversation_history.append(HumanMessage(content=user_input))
    result =agent.invoke({"messages":conversation_history})
    print(result["messages"])
    conversation_history=result["messages"]
    user_input=input("Enter: ")


AI says : Hi! Please provide the question or context so I can assist you effectively.
Current State :  [HumanMessage(content='hi, answer in english', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! Please provide the question or context so I can assist you effectively.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='hi, answer in english', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! Please provide the question or context so I can assist you effectively.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [11]:
with open("logging.txt","w") as file:
    file.write("Your conversation log :\n")
    for message in conversation_history:
        if isinstance(message,HumanMessage):
            file.write(f"You : {message.content}\n")
        elif isinstance(message,AIMessage):
            file.write(f"AI: {message.content}\n\n")
        file.write("End of conversation")
print("Conversation saved")

Conversation saved
